# Phase 2: Grad-CAM for Image Branch

Generates **target-specific Grad-CAM heatmaps** for the image branch.

| Component | Configuration |
|---|---|
| Image Backbone | Swin-B (`swin_base_patch4_window7_224`) |
| Text Backbone | PhoBERT (`vinai/phobert-base-v2`) |
| Fusion | Cross-Attention (8 heads, hidden=512) |
| XAI Method | Manual Grad-CAM via hooks |

**Key features:**
- Per-target heatmaps (5 targets)
- Per-image heatmaps for multi-image reviews (up to 4 images)
- Gradient diagnostics to explain target similarity
- 15-sample batch processing with smart selection

---
### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone and install

In [ ]:
!rm -rf /content/SE365
!git clone -b xai-v2 https://github.com/lechihoang/SE365.git /content/SE365
%cd /content/SE365
!pip install -q -r requirements.txt

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configuration

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_OUT_DIR  = f'{EXP_DIR}/xai/gradcam'
DATA_DIR     = f'{PROJECT_ROOT}/data/text'
IMAGE_DIR    = f'{PROJECT_ROOT}/data/image'

NUM_GRADCAM_SAMPLES = 15

os.makedirs(XAI_OUT_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT       : {PROJECT_ROOT}')
print(f'EXP_DIR            : {EXP_DIR}')
print(f'XAI_OUT_DIR        : {XAI_OUT_DIR}')
print(f'NUM_GRADCAM_SAMPLES: {NUM_GRADCAM_SAMPLES}')

### STEP 5: Imports and Seed

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 5 — Imports and Seed')
print('='*60)

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage

from xai.config import (
    TARGET_NAMES, FACTOR_NAMES, DISPLAY_NAMES, NUM_TARGETS,
    IMAGE_FEATURE_DIM, DEFAULT_SEED, DEFAULT_DPI,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
)
from xai.utils import (
    get_device, set_seed, get_tokenizer, get_image_processor,
    load_model, load_single_sample, get_prediction,
    save_raw_values, get_metadata,
)
from xai.gradcam_explainer import (
    GradCAMExplainer, compute_gradcam_for_image,
    overlay_cam_on_image, find_target_layer,
    create_5target_comparison, diagnose_target_gradients,
)

SEED = DEFAULT_SEED
set_seed(SEED)
device = get_device()

print(f'Device  : {device}')
print(f'Seed    : {SEED}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 6: Load Model + Eager Attention Patch

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 6 — Load Model')
print('='*60)

model, config = load_model(EXP_DIR, device=device)

# Patch sdpa -> eager attention
enc = model.text_model.encoder
if hasattr(enc, 'config'):
    enc.config._attn_implementation = 'eager'
    enc.config.attn_implementation = 'eager'
patched = 0
try:
    from transformers.models.roberta.modeling_roberta import RobertaSelfAttention
    if hasattr(enc, 'encoder') and hasattr(enc.encoder, 'layer'):
        for lm in enc.encoder.layer:
            attn = lm.attention.self
            if 'Sdpa' in type(attn).__name__ or 'Flash' in type(attn).__name__:
                ea = RobertaSelfAttention(enc.config)
                ea.load_state_dict(attn.state_dict())
                ea.to(next(attn.parameters()).device)
                lm.attention.self = ea
                patched += 1
except Exception as e:
    print(f'Attention patch skipped: {e}')
if patched > 0:
    print(f'Patched {patched} attention layers: sdpa -> eager')

print(f'Model   : {model.__class__.__name__} ({sum(p.numel() for p in model.parameters()):,} params)')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 7: Tokenizer, Processor, Data Split, Sample Selection

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 7 — Tokenizer & Sample Selection')
print('='*60)

text_model_name = config.get('text_model_name', BEST_TEXT_MODEL)
image_model_name = config.get('image_model_name', BEST_IMAGE_MODEL)
tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

test_csv = os.path.join(DATA_DIR, 'test.csv')
val_csv  = os.path.join(DATA_DIR, 'val.csv')
SPLIT_CSV = test_csv if os.path.isfile(test_csv) else val_csv
SPLIT_NAME = 'test' if SPLIT_CSV == test_csv else 'validation'

# Smart sample selection: mix of correct, high-error, and multi-image
pred_csv = os.path.join(EXP_DIR, 'test_predictions.csv' if SPLIT_NAME == 'test' else 'predictions.csv')
df_split = pd.read_csv(SPLIT_CSV)

if os.path.isfile(pred_csv):
    df_pred = pd.read_csv(pred_csv)
    error_cols = [c for c in df_pred.columns if c.startswith('absolute_error_')]
    if error_cols and len(df_pred) >= NUM_GRADCAM_SAMPLES:
        df_pred['mean_error'] = df_pred[error_cols].mean(axis=1)
        # 5 lowest error (correct), 5 highest error, 5 from middle
        sorted_idx = df_pred.sort_values('mean_error')['index' if 'index' in df_pred.columns else df_pred.index.name or 'DUMMY'].values
        if 'index' not in df_pred.columns:
            sorted_idx = df_pred.sort_values('mean_error').index.values
        n = len(sorted_idx)
        low_err = sorted_idx[:5].tolist()
        high_err = sorted_idx[-5:].tolist()
        mid = sorted_idx[n//2 - 2 : n//2 + 3].tolist()
        SAMPLE_INDICES = list(dict.fromkeys(low_err + mid + high_err))[:NUM_GRADCAM_SAMPLES]
        print(f'Smart selection: {len(low_err)} correct + {len(mid)} mid + {len(high_err)} high-error')
    else:
        SAMPLE_INDICES = list(range(min(NUM_GRADCAM_SAMPLES, len(df_split))))
        print(f'Fallback: first {len(SAMPLE_INDICES)} samples')
else:
    SAMPLE_INDICES = list(range(min(NUM_GRADCAM_SAMPLES, len(df_split))))
    print(f'No predictions CSV. Using first {len(SAMPLE_INDICES)} samples.')

# Filter to valid range
SAMPLE_INDICES = [int(i) for i in SAMPLE_INDICES if 0 <= int(i) < len(df_split)]

print(f'Split   : {SPLIT_NAME} ({len(df_split)} samples)')
print(f'Selected: {SAMPLE_INDICES}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 8: Load Demo Sample and Display

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 8 — Load Demo Sample')
print('='*60)

demo_idx = SAMPLE_INDICES[0]
sample = load_single_sample(
    csv_path=SPLIT_CSV, idx=demo_idx,
    tokenizer=tokenizer, image_processor=image_processor,
    image_dir=IMAGE_DIR, device=device,
)
result = get_prediction(model, sample)
num_images = sample['num_real_images']

print(f'Sample       : {demo_idx}')
print(f'Images       : {num_images}')
print(f'Text (80ch)  : {sample["text"][:80]}...')
print(f'\n{"Target":<28s} {"Pred":>8s} {"GT":>8s}')
print('-'*46)
for name in TARGET_NAMES:
    print(f'{name:<28s} {result["predictions"][name]:8.3f} {result["ground_truth"][name]:8.2f}')

fig, axes = plt.subplots(1, max(num_images, 1), figsize=(4*max(num_images,1), 4))
if num_images == 1: axes = [axes]
for i, (img, ax) in enumerate(zip(sample['loaded_images'], axes)):
    ax.imshow(img); ax.set_title(f'Image {i}'); ax.axis('off')
plt.suptitle(f'Sample {demo_idx}', fontsize=12)
plt.tight_layout(); plt.show()
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 9: Verify Target Layer

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 9 — Verify Target Layer')
print('='*60)

target_layer = find_target_layer(model)
print(f'Target layer : {type(target_layer).__name__}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 10: Gradient Diagnostics — Target Similarity Analysis

This is the key diagnostic: we check whether gradients from different targets
are actually different at the image encoder level. If they are very similar,
that's expected — the 5 targets share the entire encoder and fusion head,
differing only at the final Linear(256→5) layer.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 10 — Gradient Diagnostics')
print('='*60)

diag = diagnose_target_gradients(
    model=model, sample=sample,
    target_layer=target_layer, device=device, image_idx=0,
)

# Print gradient statistics
print('\n--- Gradient Statistics per Target ---')
print(f'{"Target":>10s} {"Pred":>8s} {"GradMean":>10s} {"GradStd":>10s} {"AbsMax":>10s} {"NonZero%":>8s}')
for gs in diag['grad_stats']:
    print(f'{gs["target"]:>10s} {gs["predicted_score"]:8.3f} {gs["grad_mean"]:10.2e} {gs["grad_std"]:10.2e} {gs["grad_abs_max"]:10.2e} {gs["nonzero_ratio"]*100:7.1f}%')

# Print raw CAM statistics
print('\n--- Raw CAM Statistics per Target (before normalization) ---')
print(f'{"Target":>10s} {"Min":>10s} {"Max":>10s} {"Mean":>10s} {"Std":>10s}')
for cs in diag['cam_stats']:
    print(f'{cs["target"]:>10s} {cs["cam_min"]:10.4f} {cs["cam_max"]:10.4f} {cs["cam_mean"]:10.4f} {cs["cam_std"]:10.4f}')

# Display gradient similarity matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(diag['grad_similarity_matrix'], vmin=-1, vmax=1, cmap='RdBu_r')
ax1.set_xticks(range(5)); ax1.set_yticks(range(5))
ax1.set_xticklabels(FACTOR_NAMES, fontsize=9, rotation=45)
ax1.set_yticklabels(FACTOR_NAMES, fontsize=9)
for i in range(5):
    for j in range(5):
        ax1.text(j, i, f'{diag["grad_similarity_matrix"][i,j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im1, ax=ax1)
ax1.set_title('Gradient Cosine Similarity', fontsize=11)

im2 = ax2.imshow(diag['cam_correlation_matrix'], vmin=-1, vmax=1, cmap='RdBu_r')
ax2.set_xticks(range(5)); ax2.set_yticks(range(5))
ax2.set_xticklabels(FACTOR_NAMES, fontsize=9, rotation=45)
ax2.set_yticklabels(FACTOR_NAMES, fontsize=9)
for i in range(5):
    for j in range(5):
        ax2.text(j, i, f'{diag["cam_correlation_matrix"][i,j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im2, ax=ax2)
ax2.set_title('Raw CAM Pearson Correlation', fontsize=11)

plt.suptitle(f'Sample {demo_idx} — Target Similarity Diagnosis', fontsize=12)
plt.tight_layout()

diag_path = os.path.join(XAI_OUT_DIR, f'sample_{demo_idx:04d}', 'gradient_diagnostics.png')
os.makedirs(os.path.dirname(diag_path), exist_ok=True)
fig.savefig(diag_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {diag_path}')

# Interpret
off_diag_grad = diag['grad_similarity_matrix'][np.triu_indices(5, k=1)]
off_diag_cam = diag['cam_correlation_matrix'][np.triu_indices(5, k=1)]
print(f'\nGradient similarity: mean={off_diag_grad.mean():.3f}, min={off_diag_grad.min():.3f}')
print(f'CAM correlation:     mean={off_diag_cam.mean():.3f}, min={off_diag_cam.min():.3f}')

if off_diag_grad.mean() > 0.95:
    print('\n>>> EXPECTED: Gradients are very similar across targets.')
    print('    The 5 targets share the ENTIRE image encoder and fusion head.')
    print('    Only the final Linear(256->5) layer differs per target.')
    print('    By the time gradients reach encoder.norm, target differences are diluted.')
    print('    This is NOT a bug. Use SHAP (Phase 4) for target-specific modality analysis.')
else:
    print('\n>>> Gradients show some target specificity.')

# Save diagnostic data
diag_json = {
    'sample_idx': demo_idx,
    'grad_stats': diag['grad_stats'],
    'cam_stats': diag['cam_stats'],
    'grad_similarity_mean': float(off_diag_grad.mean()),
    'grad_similarity_min': float(off_diag_grad.min()),
    'cam_correlation_mean': float(off_diag_cam.mean()),
    'cam_correlation_min': float(off_diag_cam.min()),
}
diag_json_path = os.path.join(XAI_OUT_DIR, f'sample_{demo_idx:04d}', 'gradient_diagnostics.json')
save_raw_values(diag_json, diag_json_path)

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 11: Single-Target Demo (food_score)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 11 — Single-Target Demo (food_score)')
print('='*60)

cam = compute_gradcam_for_image(
    model=model, sample=sample,
    target_idx=0, image_idx=0,
    target_layer=target_layer, device=device,
)
print(f'CAM shape: {cam.shape}, range: [{cam.min():.4f}, {cam.max():.4f}]')
assert np.isfinite(cam).all(), 'CAM contains NaN/Inf'

overlay = overlay_cam_on_image(cam, sample['loaded_images'][0])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample['loaded_images'][0].resize((224, 224))); axes[0].set_title('Original')
axes[1].imshow(cam, cmap='jet', vmin=0, vmax=1); axes[1].set_title(f'Raw CAM ({cam.shape[0]}x{cam.shape[1]})')
axes[2].imshow(overlay); axes[2].set_title('Grad-CAM: food_score')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 12: All 5 Targets for One Image

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 12 — All 5 Targets')
print('='*60)

cams_5target = []
for t_idx in range(NUM_TARGETS):
    c = compute_gradcam_for_image(
        model=model, sample=sample,
        target_idx=t_idx, image_idx=0,
        target_layer=target_layer, device=device,
    )
    cams_5target.append(c)
    print(f'  {FACTOR_NAMES[t_idx]:>10s}: range=[{c.min():.3f}, {c.max():.3f}]')

fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
axes[0].imshow(sample['loaded_images'][0].resize((224, 224)))
axes[0].set_title('Original', fontsize=10, fontweight='bold'); axes[0].axis('off')
for t_idx in range(NUM_TARGETS):
    ov = overlay_cam_on_image(cams_5target[t_idx], sample['loaded_images'][0])
    axes[t_idx+1].imshow(ov)
    axes[t_idx+1].set_title(DISPLAY_NAMES[t_idx], fontsize=9, fontweight='bold')
    axes[t_idx+1].axis('off')
plt.suptitle(f'Sample {demo_idx} — Grad-CAM: 5 Targets', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 13: Full Explanation with GradCAMExplainer

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 13 — Full Sample Explanation')
print('='*60)

explainer = GradCAMExplainer(model=model, device=device, output_dir=XAI_OUT_DIR)

sample_id = f'sample_{demo_idx:04d}'
results = explainer.explain_sample(sample=sample, sample_id=sample_id)

print(f'\nComparison figure: {results["comparison_path"]}')
print(f'Overlays: {len(results["individual_paths"])}')

comp_img = PILImage.open(results['comparison_path'])
fig, ax = plt.subplots(1, 1, figsize=(18, 4))
ax.imshow(comp_img); ax.axis('off')
ax.set_title(f'{sample_id} — 5-Target Comparison', fontsize=12)
plt.tight_layout(); plt.show()
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 14: Batch Processing — 15 Samples

In [ ]:
t0 = time.time()
print('='*60)
print(f'  Phase 2 — Step 14 — Batch Processing ({NUM_GRADCAM_SAMPLES} samples)')
print('='*60)

batch_results = []

for sidx in SAMPLE_INDICES:
    ts = time.time()
    sid = f'sample_{sidx:04d}'
    print(f'\n--- {sid} ---')
    try:
        s = load_single_sample(
            csv_path=SPLIT_CSV, idx=sidx,
            tokenizer=tokenizer, image_processor=image_processor,
            image_dir=IMAGE_DIR, device=device,
        )
        r = explainer.explain_sample(sample=s, sample_id=sid)
        elapsed = time.time() - ts
        batch_results.append({
            'sample_id': sid, 'sample_idx': sidx,
            'num_images': s['num_real_images'],
            'status': 'success', 'elapsed_s': round(elapsed, 1),
            'num_artifacts': len(r['individual_paths']),
        })
        print(f'  OK: {elapsed:.1f}s, {len(r["individual_paths"])} overlays')
    except Exception as e:
        elapsed = time.time() - ts
        batch_results.append({
            'sample_id': sid, 'sample_idx': sidx,
            'status': 'failed', 'error': str(e), 'elapsed_s': round(elapsed, 1),
        })
        print(f'  FAILED: {e}')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

batch_summary = {
    'phase': 'Phase 2: Grad-CAM',
    'experiment_id': EXP_ID, 'split': SPLIT_NAME,
    'num_samples': len(SAMPLE_INDICES),
    'total_elapsed_s': round(time.time() - t0, 1),
    'results': batch_results,
}
summary_path = os.path.join(XAI_OUT_DIR, 'gradcam_batch_summary.json')
save_raw_values(batch_summary, summary_path)

n_ok = sum(1 for r in batch_results if r['status'] == 'success')
print(f'\nBatch complete: {n_ok}/{len(batch_results)} succeeded')
print(f'Total time: {time.time()-t0:.1f}s')

### STEP 15: Reproducibility Check

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 15 — Reproducibility Check')
print('='*60)

cam_r1 = compute_gradcam_for_image(model=model, sample=sample, target_idx=0, image_idx=0, target_layer=target_layer, device=device)
cam_r2 = compute_gradcam_for_image(model=model, sample=sample, target_idx=0, image_idx=0, target_layer=target_layer, device=device)

is_identical = np.allclose(cam_r1, cam_r2, atol=1e-7)
max_diff = np.abs(cam_r1 - cam_r2).max()
print(f'Max diff   : {max_diff:.2e}')
print(f'Identical  : {is_identical}')
print(f'Reproducibility: {"PASSED" if is_identical else "FAILED"}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 16: Gradient Flow Check

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 2 — Step 16 — Gradient Flow Check')
print('='*60)

gradient_ok = True
for t_idx in range(NUM_TARGETS):
    grads = {}
    def bwd_hook(module, grad_input, grad_output):
        grads['value'] = grad_output[0].detach()
    handle = target_layer.register_full_backward_hook(bwd_hook)
    model.zero_grad()
    try:
        with torch.enable_grad():
            pv = sample['pixel_values'].clone().detach().requires_grad_(True)
            output = model(input_ids=sample['input_ids'], attention_mask=sample['attention_mask'],
                           pixel_values=pv, num_images=sample.get('num_images'))
            preds = output[0] if isinstance(output, tuple) else output
            preds[0, t_idx].backward(retain_graph=False)
        if 'value' in grads:
            gs = grads['value'].abs().sum().item()
            status = 'OK' if gs > 0 else 'ZERO'
            if gs == 0: gradient_ok = False
            print(f'  {FACTOR_NAMES[t_idx]:>10s}: grad_abs_sum={gs:.4f} [{status}]')
        else:
            print(f'  {FACTOR_NAMES[t_idx]:>10s}: NO GRADIENT'); gradient_ok = False
    finally:
        handle.remove(); model.zero_grad(); grads.clear()

print(f'\nGradient flow: {"PASSED" if gradient_ok else "FAILED"}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 17: Final Summary

In [ ]:
print('='*60)
print('  PHASE 2 GRAD-CAM — FINAL SUMMARY')
print('='*60)

artifact_count = sum(len(f) for _, _, f in os.walk(XAI_OUT_DIR))
n_ok = sum(1 for r in batch_results if r['status'] == 'success')

# Target specificity from step 12
flat_cams = [c.flatten() for c in cams_5target]
corr_mat = np.zeros((5,5))
for i in range(5):
    for j in range(5):
        if np.std(flat_cams[i]) > 1e-8 and np.std(flat_cams[j]) > 1e-8:
            corr_mat[i,j] = np.corrcoef(flat_cams[i], flat_cams[j])[0,1]
        else:
            corr_mat[i,j] = 1.0 if i==j else 0.0
off_d = corr_mat[np.triu_indices(5, k=1)]
specificity_ok = off_d.min() < 0.99

print(f'  Experiment      : {EXP_ID}')
print(f'  Split           : {SPLIT_NAME}')
print(f'  Samples         : {n_ok}/{NUM_GRADCAM_SAMPLES} succeeded')
print(f'  Total artifacts : {artifact_count}')
print(f'  Output dir      : {XAI_OUT_DIR}')
print()

checks = [
    ('Model loaded',        True),
    ('Target layer found',  target_layer is not None),
    ('Single CAM valid',    cam is not None and cam.shape[0] > 1),
    ('5-target comparison', len(cams_5target) == 5),
    ('Reproducibility',     is_identical),
    ('Gradient flow',       gradient_ok),
    ('Batch processing',    n_ok == len(SAMPLE_INDICES)),
    ('Diagnostics saved',   os.path.isfile(diag_json_path)),
]

all_passed = True
for desc, passed in checks:
    s = 'PASSED' if passed else 'FAILED'
    if not passed: all_passed = False
    print(f'  [{s:6s}] {desc}')

# Note about target similarity
print(f'\n  CAM correlation (off-diag): mean={off_d.mean():.3f}, min={off_d.min():.3f}')
if off_d.mean() > 0.9:
    print('  NOTE: Heatmaps are similar across targets — this is expected.')
    print('        The image encoder is shared; target differences are subtle.')
    print('        Use SHAP (Phase 4) for target-specific modality analysis.')

print('='*60)
if all_passed:
    print('  All checks PASSED. Phase 2 complete.')
else:
    print('  Some checks FAILED.')
print('='*60)